# Campaign CSV Bootstrap Import Demo

This notebook demonstrates `solsys_code/management/commands/import_campaign_csv.py`
(Phase 14, CAMP-04/CAMP-05), the bootstrap-import command that turns a campaign
coordination CSV (e.g. the real 3I/ATLAS Google Sheet) into `CampaignRun` rows.

**The fixture used here (`fixtures/campaign_sample.csv`) is entirely synthetic and
PII-free** (CAMP-05) — every contact name is a placeholder (`Ada Test`, `Ben Sample`, ...)
and every email address is `@example.com`/`@example.org`. No real contact information
from the actual 3I/ATLAS coordination sheet is ever read by this notebook.

It demonstrates:

- Seeding `Observatory` records for the 3 sites used by the fixture (idempotent
  `update_or_create`), so site resolution hits the local database only — no live
  MPC Obscodes API call happens anywhere in this notebook (D-11)
- Seeding a single-`Target` campaign `TargetList` so the auto-target-resolution
  behavior (D-07/CAMP-02) is exercised
- Invoking `import_campaign_csv` via `call_command` and inspecting the printed
  created/updated/unchanged/skipped/site_needs_review summary
- Inspecting the resulting `CampaignRun` rows
- The `pending_review` -> `approved`/`rejected` `approval_status` lifecycle (D-03)
  on synthetic data — the bootstrap import itself always writes `approved`
  (vetted historical backfill), so this lifecycle has no other demonstration path
- Re-running the command to confirm idempotency (no duplicate rows)

This notebook lives in `pre_executed/` because it is **DB-dependent** (it seeds
`Observatory`/`TargetList` records and creates `CampaignRun` rows) and is therefore
**NOT run during Sphinx/CI/ReadTheDocs builds**, per `docs/notebooks/README.md`.

## Django setup

Standard boilerplate to make `src.fomo.settings` importable from this notebook's
location (`docs/notebooks/pre_executed/` — three levels under the repo root, so
`parents[2]` gives the repo root) and to allow synchronous ORM calls inside
Jupyter's async event loop.

In [1]:
import os
import sys
from pathlib import Path

import django

# Ensure the repo root is on sys.path so `src.fomo.settings` is importable
# when this notebook is executed from docs/notebooks/pre_executed/.
# NOTE: parents[2] is correct only when the Jupyter kernel CWD is
# docs/notebooks/pre_executed/. Start Jupyter from that directory, or
# adjust the index if you launch from the repo root.
repo_root_path = Path.cwd().resolve().parents[2]
assert (
    repo_root_path / 'manage.py'
).exists(), (
    f'Repo root not found at {repo_root_path}. Run Jupyter from docs/notebooks/pre_executed/ or adjust parents[] index.'
)
repo_root = str(repo_root_path)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'src.fomo.settings')

# Jupyter's ipykernel runs inside an asyncio event loop, but Django's ORM is
# sync-only by default and refuses to run there; this opts back in.
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

django.setup()

# This notebook intentionally imports only the campaign-coordination models and
# the observatory package below -- never the ephemeris view/computation modules,
# which trigger a large one-time SPICE kernel download on first import.

## Seed Observatory records and the campaign TargetList

`import_campaign_csv` resolves each row's `Site Code` against the `Observatory`
model first (tier 1 of the D-08 3-tier resolution); only a tier-1 miss falls
through to a live MPC Obscodes API call. The fixture's three non-blank Site
Codes (`F65`, `309`, `705`) are seeded here first, so every row in this
notebook resolves locally and no network call happens (D-11). `update_or_create`
makes this cell idempotent — safe to re-run against any dev DB.

The campaign container is a `tom_targets.models.TargetList` — found-or-created by
name (D-06). Linking exactly one `Target` to it (via `NonSiderealTargetFactory`,
per this project's Target-factory convention — FOMO is exclusively for Solar
System / non-sidereal targets) demonstrates auto-target resolution: every
imported `CampaignRun` gets that `Target` assigned automatically (D-07/CAMP-02).

In [2]:
from tom_targets.models import TargetList
from tom_targets.tests.factories import NonSiderealTargetFactory

from solsys_code.solsys_code_observatory.models import Observatory

SITE_DATA = {
    'F65': dict(name='Faulkes Telescope North', short_name='FTN', lat=20.7, lon=-156.3, altitude=3055),
    '309': dict(
        name='European Southern Observatory, Paranal', short_name='VLT', lat=-24.6272, lon=-70.4039, altitude=2635
    ),
    '705': dict(name='Apache Point Observatory', short_name='APO', lat=32.7803, lon=-105.8203, altitude=2788),
}

for obscode, fields in SITE_DATA.items():
    obs, created = Observatory.objects.update_or_create(obscode=obscode, defaults=fields)
    action = 'created' if created else 'updated/unchanged'
    print(f'  obscode={obscode!r:>4}  {action:>18}  short_name={obs.short_name!r}')

campaign, campaign_created = TargetList.objects.get_or_create(name='3I/ATLAS (demo)')
print(f'\nCampaign TargetList {campaign.name!r} {"created" if campaign_created else "found"}')

if campaign.targets.count() == 0:
    demo_target = NonSiderealTargetFactory.create(name='3I/ATLAS (demo target)')
    campaign.targets.add(demo_target)
    print(f'  Linked new Target: {demo_target.name!r}')
else:
    print(f'  Already has {campaign.targets.count()} linked Target(s)')

print(f'Campaign now has {campaign.targets.count()} linked Target(s): {[t.name for t in campaign.targets.all()]}')

Target post save hook: 3I/ATLAS (demo target) created: True


Target post save hook: 3I/ATLAS (demo target) created: False


  obscode='F65'             created  short_name='FTN'
  obscode='309'             created  short_name='VLT'
  obscode='705'             created  short_name='APO'

Campaign TargetList '3I/ATLAS (demo)' created
  Linked new Target: '3I/ATLAS (demo target)'
Campaign now has 1 linked Target(s): ['3I/ATLAS (demo target)']


## The synthetic fixture

`fixtures/campaign_sample.csv` has the same 14-column shape as the real
3I/ATLAS coordination sheet, but every row is hand-built synthetic data (D-10):
placeholder contact names, `@example.com`/`@example.org` emails, and only the
three `Site Code` values seeded above. It covers:

- a clean multi-band imaging row (`griz`, a normal `HH:MM - HH:MM` UT range)
- a spectroscopy row (`Open to collaboration? = yes`)
- an `Observation Status` that maps to a terminal `run_status` (`cancelled`)
- an approximate UT time (`~1 am`) and a fully blank UT time, exercising both
  of `parse_obs_window`'s best-effort fallback paths
- a blank `Site Code`, exercising the `site_needs_review` flag with no
  Observatory match attempted

In [3]:
import io

from django.core.management import call_command

fixture_path = repo_root_path / 'docs' / 'notebooks' / 'pre_executed' / 'fixtures' / 'campaign_sample.csv'
assert fixture_path.exists(), f'Fixture not found at {fixture_path}'

stdout_buf = io.StringIO()
stderr_buf = io.StringIO()

call_command(
    'import_campaign_csv',
    '--campaign',
    '3I/ATLAS (demo)',
    str(fixture_path),
    stdout=stdout_buf,
    stderr=stderr_buf,
)

print('stdout:', stdout_buf.getvalue())
if stderr_buf.getvalue():
    print('stderr (skipped rows):', stderr_buf.getvalue())

stdout: Done. created: 6, updated: 0, unchanged: 0, skipped: 0, site_needs_review: 1



## Inspect the imported CampaignRun rows

Confirm the summary above matches: 6 rows in the fixture, all with a resolvable
`Telescope / Instrument`/`Obs. Date`, so all 6 should be `created` on a first
run, with exactly one `site_needs_review` (the blank-Site-Code row). No real
contact information appears below — only the synthetic placeholders from the
fixture.

In [4]:
from solsys_code.models import CampaignRun

runs = CampaignRun.objects.filter(campaign=campaign).order_by('obs_date', 'ut_start')
print(f'Total CampaignRun rows for {campaign.name!r}: {runs.count()}')
print()
header = f'{"telescope_instrument":<34} {"run_status":<10} {"site_needs_review":<18} {"target":<26}'
print(header)
print('-' * len(header))
for run in runs:
    target_label = run.target.name if run.target else '(none)'
    print(f'{run.telescope_instrument:<34} {run.run_status:<10} {str(run.site_needs_review):<18} {target_label:<26}')

Total CampaignRun rows for '3I/ATLAS (demo)': 6

telescope_instrument               run_status site_needs_review  target                    
-------------------------------------------------------------------------------------------
ESO VLT FORS2                      observed   False              3I/ATLAS (demo target)    
FTN/MuSCAT3                        observed   False              3I/ATLAS (demo target)    
Apache Point Observatory/KOSMOS    observed   False              3I/ATLAS (demo target)    
Apache Point Observatory/ARCTIC    cancelled  False              3I/ATLAS (demo target)    
Generic 1m robotic telescope       planned    True               3I/ATLAS (demo target)    
VLT/MUSE                           observed   False              3I/ATLAS (demo target)    


## Approval lifecycle: pending_review -> approved / rejected (D-03)

Bootstrap-imported rows always land with `approval_status=APPROVED` — they are
vetted historical data being backfilled, not fresh submissions awaiting review
(D-03). The full `pending_review` -> `approved`/`rejected` lifecycle only
becomes operationally relevant once Phase 16's community submission form
exists. It is demonstrated here directly on two synthetic `CampaignRun` rows,
created outside the CSV import, so the transition has automated/notebook
coverage somewhere in this milestone.

In [5]:
pending_run = CampaignRun.objects.create(
    campaign=campaign,
    telescope_instrument='Demo Telescope/DemoCam',
    contact_person='Grace Lifecycle',
    contact_email='grace.lifecycle@example.com',
    approval_status=CampaignRun.ApprovalStatus.PENDING_REVIEW,
)
print(f'Created (approval demo #1): approval_status={pending_run.approval_status!r}')

pending_run.approval_status = CampaignRun.ApprovalStatus.APPROVED
pending_run.save(update_fields=['approval_status'])
pending_run.refresh_from_db()
print(f'After staff approval:       approval_status={pending_run.approval_status!r}')

print()

rejected_run = CampaignRun.objects.create(
    campaign=campaign,
    telescope_instrument='Demo Telescope/DemoSpec',
    contact_person='Hal Lifecycle',
    contact_email='hal.lifecycle@example.com',
    approval_status=CampaignRun.ApprovalStatus.PENDING_REVIEW,
)
print(f'Created (approval demo #2): approval_status={rejected_run.approval_status!r}')

rejected_run.approval_status = CampaignRun.ApprovalStatus.REJECTED
rejected_run.save(update_fields=['approval_status'])
rejected_run.refresh_from_db()
print(f'After staff rejection:      approval_status={rejected_run.approval_status!r}')

Created (approval demo #1): approval_status=CampaignRun.ApprovalStatus.PENDING_REVIEW
After staff approval:       approval_status='approved'

Created (approval demo #2): approval_status=CampaignRun.ApprovalStatus.PENDING_REVIEW
After staff rejection:      approval_status='rejected'


## Idempotency check

Running the command a second time with the same fixture should produce zero
new rows and zero updates — the summary should report `created: 0, updated: 0,
unchanged: 6`. The two synthetic approval-lifecycle rows above are untouched
(different `telescope_instrument` values, not present in the fixture).

In [6]:
stdout_buf2 = io.StringIO()
stderr_buf2 = io.StringIO()

call_command(
    'import_campaign_csv',
    '--campaign',
    '3I/ATLAS (demo)',
    str(fixture_path),
    stdout=stdout_buf2,
    stderr=stderr_buf2,
)

print('Second run stdout:', stdout_buf2.getvalue())
print(
    f'Total CampaignRun rows for {campaign.name!r} after re-run: '
    f'{CampaignRun.objects.filter(campaign=campaign).count()}'
)

Second run stdout: Done. created: 0, updated: 0, unchanged: 6, skipped: 0, site_needs_review: 1

Total CampaignRun rows for '3I/ATLAS (demo)' after re-run: 8


## Summary

This notebook demonstrates the Phase 14 Plan 03 deliverable (CAMP-05) and
exercises CAMP-04's import command end-to-end against a synthetic fixture:

| Requirement | Description | Demonstrated by |
|-------------|-------------|------------------|
| CAMP-04 | Command reports created/updated/unchanged/skipped/site_needs_review; idempotent re-run | Import cell summary (6 created) and re-run cell summary (6 unchanged) |
| CAMP-02 / D-07 | Optional `target` FK auto-resolves for a single-Target campaign | Inspection table's `target` column shows the same demo Target for every row |
| D-08 / D-09 | Site resolution never skips a row; unresolved site is flagged, not fabricated | Inspection table shows `site_needs_review=True` for the blank-Site-Code row only |
| D-03 / CAMP-03 | `pending_review` -> `approved`/`rejected` approval lifecycle | Approval-lifecycle cell before/after prints |
| CAMP-05 | No real PII in git history; no live network call | Every contact/email value above is a synthetic placeholder; Site Codes are limited to the three Observatory rows seeded locally in this notebook |

This notebook is **pre-executed** and intentionally excluded from automated doc
builds (not referenced in `docs/notebooks.rst`) because it depends on
`Observatory`/`TargetList`/`CampaignRun` records and writes to the local dev DB.